# Outline

2022/05/29-

- [X] Common types of advanced trees and basic knowledge points
    - [X] [Segment Tree](https://books.halfrost.com/leetcode/ChapterTwo/Segment_Tree/) 
    - [X] Trie
- [ ] Time/space complexity analysis
- [ ] Classic LeetCode example problems
- [ ] My summary
- [ ] Real-world applications


Reference links:
- [leetcode cookbook tree](https://books.halfrost.com/leetcode/ChapterTwo/Tree/)


#  Theoretical Knowledge
- [AVL Tree](https://cloud.tencent.com/developer/article/1762718): a binary search tree where the height difference between the left and right subtrees is at most 1. https://blog.csdn.net/qq_25940921/article/details/82183093
    - How is balance maintained? The cause of imbalance: inserting a new element. So adjustments are made after insertion — complex rotation operations. There are four cases: left rotation, right rotation, left+right rotation, right+left rotation
    - Search: complexity is O(logN)
    - Insertion: just the insertion itself is O(1) — a normal insertion plus at most two rotations. Looking at the whole process, it's O(logN)
    - Deletion: complexity is O(log(n)); the number of rotation operations can reach log(n).
    
- [Trie](https://blog.csdn.net/weixin_40374341/article/details/94028364): a prefix tree. Each node stores one character. Operations
    - Add a word: to distinguish whether a node marks the end of a word, TrieNode can have an added variable `end`, indicating the number of words ending at that letter.
    - Delete a word: if a node's `path` is 0, simply delete it directly — no need to traverse its children one by one.
    - Check whether a word exists
    - Return the number of words matching a prefix
- [Binary Indexed Tree (Fenwick Tree)](https://leetcode.cn/problems/range-sum-query-mutable/solution/shu-zhuang-shu-zu-xian-duan-shu-pythonsh-tjn5/)
    - Idea: organize the array as a tree to quickly compute range sums. Odd-numbered nodes (whose index has lowbit 1, indices starting from 1) correspond to values in the original array. If an index's lowbit is n, that node represents the sum of the n numbers ending at that index (inclusive). A number's lowbit is the value corresponding to the lowest set bit (the first 1) in its binary representation, computed as `x&-x`.
    - Properties: 
        - The size of the tree is the same as the size of the array (for convenience of index calculation, one extra slot can be added, leaving index 0 unused)
        - `tree[i] = num[i] + num[i-1] + ... + num[i-n+1], n=lowbit(i)`
        - `tree[i] = num[i] + \sum_j tree[j], j: j+lowbit(j) = i`
        - `preSum[i] = tree[i] + tree[i2=i-lowbit(i)] + tree[i3=i2-lowbit(i2)] + ... (ik >=0)`, meaning: the prefix sum at index n equals the value of the tree node at index n, plus the value of the previous tree node, plus the one before that, and so on until there are no earlier tree nodes. For a tree node at index k, the index of its preceding tree node is computed as k-lowbit(k)
    - Template: build the tree's leaf nodes starting from index 1, constructing bottom-up. The last node of the tree is the largest parent node.
        - Construction: initialize the tree using the array plus a leading 0. Iterate over the array, adding each value to the corresponding parent tree node above it; if the current index is i, the corresponding parent tree node's index is `i+lowbit(i)`
        - Updating the value at a given index:
            1. First find the original value at this index, i.e. the sum over the range [i, i]
            2. Compute the delta based on the new value,
            3. Then propagate upward, applying this delta to every parent node.
        - Computing the sum over a given range: using prefix sums, find the prefix sum at the range's right endpoint and the prefix sum at the position before the range's left endpoint, then subtract.
        - Computing a prefix sum: starting from that index, repeatedly find the root of the preceding tree and add up the values.
            - What is the "root of the preceding tree"? After removing the root that connects this node to the preceding nodes, it's the topmost root corresponding to the preceding nodes.
            - How to find it? For index j, the index of the preceding such root is j-lowbit(j), and so on.
    - Complexity: both update and query on a Binary Indexed Tree are O(logn)
        
- [Segment Tree](https://books.halfrost.com/leetcode/ChapterThree/Segment_Tree/): a binary tree where each node represents an interval. The interval represented by a parent node is the union of its children's intervals.
     - [X] How to represent an interval? Assume the data is stored in an arr[] of size n. (What if this assumption doesn't hold?)
        - The root of the segment tree usually represents the entire data range. Here that is arr[0:n-1].
        - Each leaf of the tree represents a range containing only a single element. So the leaves represent arr[0], arr[1], and so on up to arr[n-1].
        - Internal nodes of the tree represent the merged (e.g. summed) or unioned result of their children.
        - Each child node represents roughly half of the range represented by its parent. (the idea of binary division)
     - Idea: use an array, build the tree top-down, with leaves from left to right corresponding to the original array's values.
         - The root of the segment tree comes first and unfolds **top-down**, unlike a Binary Indexed Tree,
         - A Binary Indexed Tree is built **bottom-up**, with the root stored at the end of the array
     - Template:
         - Construction: build top-down. Each node represents a given contiguous interval, with left endpoint l and right endpoint r (both inclusive).
             - When l=r, a leaf node has been found — construction for this branch ends, and the result is returned to the level above
             - When l<r, recursively construct the two children, take their results, and "merge" them into this node's result
         - Updating the value at a given index: update bottom-up — first update the leaf's value, then propagate upward step by step
         - Querying the "merged" value of a given range:
             - Determine the relationship between the given range and the current node's range, and decide whether to query the left child, the right child, or "merge" the results of both children.
         - The "merge" here could be, for example, summation, taking the max, etc. It's required to satisfy an additive-like property.
     
     - [X] Why should the number of segment tree nodes be set to 4 times the array length? [Answer](https://blog.csdn.net/smoggyxhdz/article/details/78895672)
     - [ ] What properties must the "merge" operation satisfy?
     - Use cases: efficiently handling queries over an available data range or interval in logarithmic time. For example: finding points within a certain distance of the origin. There could be a large number of points in a space at various distances from a center/origin reference point. An ordinary lookup table (say a hash map) would need a linear scan over all possible points or all possible distances. A segment tree lets us achieve this in logarithmic time with much less space. This kind of problem is called **planar range search**
     - [ ] What if the array is dynamic?
     - Complexity: both update and query on a segment tree are O(logn)
    


In [ ]:
# Trie Implementation
# 208 Supports: insert, search, and search-prefix operations
class TrieNode:
    def __init__(self, val=None):
        self.val = val
        self.end = 0
        self.path = 0
        self.child = {}
        if val:
            self.path = 1

    def addChild(self, val):
        self.path += 1
        if val in self.child:
            child_node = self.child[val]
            child_node.path += 1
        else:
            child_node = TrieNode(val)
            self.child[val] = child_node

class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word: str) -> None:
        head = self.root
        for c in word:
            head.addChild(c)
            head = head.child[c]
        head.end += 1

    def search(self, word: str) -> bool:
        head = self.root
        for c in word:
            if c not in head.child:
                return False
            head = head.child[c]
        return head.end > 0

    def startsWith(self, prefix: str) -> bool:
        head = self.root
        for c in prefix:
            if c not in head.child:
                return False
            head = head.child[c]
        return True



# Your Trie object will be instantiated and called as such:
# obj = Trie()
# obj.insert(word)
# param_2 = obj.search(word)
# param_3 = obj.startsWith(prefix)

In [ ]:
# Trie Implementation
# 208 Supports: insert, search, and search-prefix operations. The delete function was added by myself
class TrieNode:
    def __init__(self, val=None):
        self.val = val
        self.end = 0 # tracks the number of words ending at this letter
        self.path = 0 # tracks the number of words that pass through this letter at this position
        self.child = {} # child nodes of this letter
        if val:
            self.path = 1

    def addChild(self, val):
        self.path += 1 # increment this letter's path by 1
        if val in self.child:
            child_node = self.child[val]
            child_node.path += 1 # if a child for this letter already exists, increment its path by 1
        else:
            child_node = TrieNode(val)
            self.child[val] = child_node

class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word: str) -> None:
        head = self.root
        for c in word:
            head.addChild(c)
            head = head.child[c]
        head.end += 1 # increment the end count of the ending letter by 1

    def search(self, word: str) -> bool:
        head = self.root
        for c in word:
            if c not in head.child:
                return False
            head = head.child[c]
        return head.end > 0 # check whether any word ends at this letter

    def startsWith(self, prefix: str) -> bool:
        head = self.root
        for c in prefix:
            if c not in head.child:
                return False
            head = head.child[c]
        return True
    
    def delete(self, word: str): # added delete functionality
        if not self.search(word): # first check whether the word exists in the trie. If not, don't delete
            return
        head = self.root
        for c in word:
            child = head.child[c]
            if child.path == 1:
                head.child.pop(c)
                break
            else:
                child.path -= 1
                head = child
        head.end -= 1

# Your Trie object will be instantiated and called as such:
# obj = Trie()
# obj.insert(word)
# param_2 = obj.search(word)
# param_3 = obj.startsWith(prefix)

In [ ]:
# Binary Indexed Tree template https://leetcode.cn/problems/range-sum-query-mutable/solution/shu-zhuang-shu-zu-xian-duan-shu-pythonsh-tjn5/

'''BIT: Binary Indexed Tree'''
class NumArray:
    
    '''custom lowbit function'''
    def lowbit(self, x: int) -> int:
        return x&(-x)

    def __init__(self, nums: List[int]):
        self.tree = [0] + nums              # the constructed BIT has one more slot than nums; index 0 is unused
        for i in range(1, len(self.tree)):  # building the BIT this way has time complexity O(n)
            j = i + self.lowbit(i)          # add the current tree node's value to its corresponding parent tree node
            if j < len(self.tree):
                self.tree[j] += self.tree[i]

    def update(self, index: int, val: int) -> None:
        pre_val = self.sumRange(index, index)   # first find the original value at this index, i.e. the sum over [i, i]
        delta = val - pre_val           # the change in value
        i = index + 1                   # i: the position of this value within the BIT (index+1)
        while i < len(self.tree): # propagate upward, applying this delta to every parent node.
            self.tree[i] += delta
            i += self.lowbit(i) # find the parent node one level up

    
    def sumRange(self, left: int, right: int) -> int:
        return self.preSum(right) - self.preSum(left-1)   # prefix sum at right minus prefix sum at left-1

    '''custom prefix-sum function preSum'''
    def preSum(self, index: int) -> int:    # index: position in the original nums
        i = index + 1                       # i: the position of this value within the BIT (index+1)
        summ = 0
        while i: # starting from this index, repeatedly find the root of the preceding tree and sum the values.
            summ += self.tree[i]
            i -= self.lowbit(i) # find the index of the preceding tree's root
            # i &= i-1        # equivalent to the line above
        return summ
    



In [ ]:
# Segment tree template

class NumArray:

    def __init__(self, nums: List[int]):
        self.len = len(nums)
        self.tree = [None] * 4 * self.len
        self.nums = nums
        self.nid2tid = [None] * self.len # store the mapping between array indices and leaf-node indices in the tree
        self.build(0, 0, self.len-1) # build top-down
    
    def build(self, i, l, r):
        if l == r: # when l=r, a leaf node has been found; construction ends here and the result is returned to the level above
            self.tree[i] = self.nums[l]
            self.nid2tid[l] = i
            return 
        mid = (l + r ) // 2
        # when l<r, recursively build the two children, take their results, and "merge" them into this node's result
        self.build(2 * i + 1, l, mid)
        self.build(2 * i + 2, mid + 1, r)
        self.tree[i] = self.tree[2*i+1] + self.tree[2*i+2]

    def query(self, tid, left, right, tleft, tright):
        if left == tleft and right == tright: # exactly the range this node is responsible for, return directly
            return self.tree[tid]
        tmid = (tleft + tright) // 2
        if right <= tmid:
            return self.query(2*tid + 1, left, right, tleft, tmid)  # this is the left subtree's range, recurse into the left subtree
        if left >= tmid + 1:
            return self.query(2*tid + 2, left, right, tmid + 1, tright) # this is the right subtree's range, recurse into the right subtree
        # left and right each cover part of the range; query both separately and merge
        return self.query(2*tid + 1, left, tmid, tleft, tmid) + self.query(2*tid + 2, tmid + 1, right, tmid + 1, tright)

    def update(self, index: int, val: int) -> None:
        diff = val - self.nums[index]
        nid = self.nid2tid[index] # find the leaf node's index,
        while nid >= 0: # update bottom-up
            self.tree[nid] += diff
            nid = (nid + 1) // 2 - 1
        self.nums[index] = val # don't forget to update the value in the array, to make computing delta easy
        
    def sumRange(self, left: int, right: int) -> int:
        return self.query(0, left, right, 0, self.len-1)

# LeetCode Example Problems

## Trie

- [(medium) 208 Implement Trie (Prefix Tree)](https://leetcode.cn/problems/implement-trie-prefix-tree/)
- [(medium) 211 Design Add and Search Words Data Structure](https://leetcode.cn/problems/design-add-and-search-words-data-structure/)
    - A few questions:
        - [X] How does the word length affect the search strategy? If the word is very long, isn't DFS better? BFS feels like it could easily run out of memory (OOM)
        - [X] How does the number of '.' characters affect the search strategy? If there are many dots, that means there are quite a few branches to explore, so DFS could be slow; if there are few dots, DFS has fewer possibilities to explore and is a good choice.
    So, if the word is relatively short, I'd go with BFS. If the word is relatively long but has few dots, I'd go with DFS.
    I implemented both search strategies, BFS and DFS.
    - Approach: designed both a BFS and a DFS version, choosing between them based on word length.

[Gong Shuisanye](https://mp.weixin.qq.com/s?__biz=MzU4NDE3MTEyMA==&mid=2247488490&idx=1&sn=db2998cb0e5f08684ee1b6009b974089&chksm=fd9cb8f5caeb31e3f7f67dba981d8d01a24e26c93ead5491edb521c988adc0798d8acb6f9e9d&token=1006889101&lang=zh_CN#rd)
- [ ] When there's time, look into how inverted indexes work

In [ ]:
class WordNode:
    def __init__(self):
        self.end = False
        self.children = {}

class WordDictionary:
    def __init__(self):
        self.root = WordNode()

    def addWord(self, word: str) -> None:
        head = self.root
        for l in word:
            if l not in head.children:
                head.children[l] = WordNode()
            head = head.children[l]
        head.end = True

    def bfsSearch(self, word):
        next_layer = [self.root]
        for l in word:
            cur_layer = next_layer
            next_layer = []
            if l == '.':
                for node in cur_layer:
                    for key, child in node.children.items():
                        next_layer.append(child)
            else:
                for node in cur_layer:
                    if l in node.children:
                        next_layer.append(node.children[l])
            if not next_layer:
                return False
        for node in next_layer:
            if node.end:
                return True
        return False

    def dfsSearch(self, word, node):
        last_l = word[0]
        if len(word) == 1:
            if last_l == '.':
                for key, child in node.children.items():
                    if child.end > 0:
                        return True
            elif last_l in node.children and node.children[last_l].end > 0:
                return True
            return False
        if last_l == '.':
            for key, child in node.children.items():
                if self.dfsSearch(word[1:], child):
                    return True
        elif last_l in node.children:
            return self.dfsSearch(word[1:], node.children[last_l])
        return False

    def search(self, word: str) -> bool:
        if len(word) > 10:
            return self.dfsSearch(word, self.root)
        return self.bfsSearch(word)



# Your WordDictionary object will be instantiated and called as such:
# obj = WordDictionary()
# obj.addWord(word)
# param_2 = obj.search(word)

## Segment Tree / Binary Indexed Tree


- [(easy) 933 Number of Recent Calls](https://leetcode.cn/problems/number-of-recent-calls/)
    - Approach 1: use a queue — record access times, and pop from the front whenever it has expired. Return the queue length as the result for this query. The downside of this approach is that it can't answer queries at an arbitrary point in time; t must be strictly increasing. What if t isn't strictly increasing?
    - [ ] Approach 2: learn the segment tree solution. First study problem 307. The solution involves something called "dynamic node creation", which looks too advanced for now — revisit later

- [(medium) 307 Range Sum Query Mutable](https://leetcode.cn/problems/range-sum-query-mutable/)
    - Approach 1: the most direct idea — just sum directly. What about the complexity? The complexity of summing is determined by left and right and can reach O(n). Submission result: TLE
    - Optimized version of Approach 1: [from a solution](https://leetcode.cn/problems/range-sum-query-mutable/solution/by-jam007-oyye/): keep the number of elements summed each time to fewer than n/2.
    - Approach 2: prefix sums. Getting a range sum is O(1), but updating a node can cost up to O(n). Submission result: TLE
    - Approach 3: Binary Indexed Tree. See [solution](https://leetcode.cn/problems/range-sum-query-mutable/solution/guan-yu-ge-lei-qu-jian-he-wen-ti-ru-he-x-41hv/); both insertion and query are O(logn)
    - Approach 4: Segment tree. Both insertion and query have time complexity O(logn)

- [(hard) 315 Count of Smaller Numbers After Self](https://leetcode.cn/problems/count-of-smaller-numbers-after-self/)
    - Approach 1: the most direct — nested loops. For each number, look at everything after it and count how many numbers are smaller. Time complexity is O(n^2). TLE.
    - Approach 2: reverse traversal + binary search + sorting. Since we need to count elements to the right, we can traverse in reverse, starting from the rightmost element. When we reach the current element, we want to find how many numbers smaller than it exist among the elements already traversed — that count is this element's result. We can use binary search to find the position, then insert this number there.
    - Approach 2 can be solved directly using Python's SortedList, [see solution](https://leetcode.cn/problems/count-of-smaller-numbers-after-self/solution/4chong-jie-fa-yi-wang-da-jin-pai-xu-shu-5vvds/).
    - [X] 💡Approach 3, to learn: merge sort. First study this [example problem](https://leetcode-cn.com/problems/shu-zu-zhong-de-ni-xu-dui-lcof/solution/4chong-jie-fa-yi-wang-da-jin-you-xu-shu-0nia5/), see the mini-topics notebook.
    - Approach 4: Binary Indexed Tree. When using a BIT, it's best to know all the tree's leaf nodes in advance, then build the tree.
        - How do we build the tree here? Instead of building it directly on the original nums array, use each element's value as the index, and store the current count of occurrences of that value at each position.
        - How do we use the BIT to get the desired result? Still traverse in reverse; finding the count of numbers smaller than the current one to its right is equivalent to querying the prefix sum of counts for values smaller than it.
        - A small trick: **coordinate compression**: when the range of values is very large (10^8) while the actual amount of data is comparatively small (10^5), and we only care about relative order rather than the exact values, we can use the relative order `rank` in place of the raw value. That is, for each number, we find its rank in the original array and build the tree over the ranks.
    - Approach 5: Segment tree. Same idea as the BIT. Compared to the official solution's segment tree, my segment tree has redundant time and space overhead in building the tree and determining parent/child indices. Let's see how the official solution does it.
    - 🌟🌟🌟Approach 5: official solution version. The way it builds the tree and computes sums is so clever!!!


    The classic array-based implementation style of the segment tree. The pushUp logic for merging two nodes is abstracted out, allowing arbitrary operations to be implemented (common operations include addition, max, min, etc). Problems 218, 303, 307, 699.
    The classic style for a counting segment tree. Problems 315, 327, 493.
    The node-based (tree-object) implementation style of the segment tree. Problems 715, 732.
    Interval lazy propagation. Problems 218, 699.
    Coordinate compression. There's a special case to watch out for with coordinate compression: suppose three intervals are [1,10], [1,4], [6,10]; after compression, x[1]=1, x[2]=4, x[3]=6, x[4]=10. The first interval becomes [1,4], the second becomes [1,2], the third becomes [3,4]. This makes interval 1 = interval 2 + interval 3, which doesn't match the pre-compression model — before compression, clearly interval 1 > interval 2 + interval 3. The correct approach is to insert an extra number between any two values whose difference is greater than 1; for example, inserting 5 between 4 and 6 above gives x[1]=1, x[2]=4, x[3]=5, x[4]=6, x[5]=10. After this, interval 1 is 1-5, interval 2 is 1-2, and interval 3 is 4-5.
    Flexible segment tree construction. A segment tree node can store multiple pieces of information, and the pushUp operation for merging two nodes can vary as well. Problems 850, 1157.

- [(hard)218 The Skyline Problem](https://leetcode.cn/problems/the-skyline-problem/)
Problem types from easy to hard:

### Single-point updates
- [HDU 1166 Enemy Troop Deployment](http://acm.hdu.edu.cn/showproblem.php?pid=1166) update: single-point add/subtract, query: range sum
- [HDU 1754 I Hate It update](http://acm.hdu.edu.cn/showproblem.php?pid=1754): single-point replace, query: range min/max
- [HDU 1394 Minimum Inversion Number update](http://acm.hdu.edu.cn/showproblem.php?pid=1394): single-point add/subtract, query: range sum
- [HDU 2795 Billboard query: find the position achieving the range max](http://acm.hdu.edu.cn/showproblem.php?pid=2795); the update operation is folded directly into the query

### Range updates
- [HDU 1698 Just a Hook](http://acm.hdu.edu.cn/showproblem.php?pid=1698) update: range replace (since the whole range is only queried once, we can just output node 1's info directly)
- [POJ 3468 A Simple Problem with Integers](http://poj.org/problem?id=3468) update: range add/subtract, query: range sum
- [POJ 2528 Mayor's posters](http://poj.org/problem?id=2528) coordinate compression + update: range replace, query: simple hash
- [POJ 3225 Help with Intervals](http://poj.org/problem?id=3225) update: range replace, range XOR, query: simple hash

### Range merging
This category of problem asks for the longest contiguous sub-range satisfying some condition, so during PushUp the left and right children's ranges need to be merged:
- [POJ 3667 Hotel](http://poj.org/problem?id=3667) update: range replace, query: find the leftmost position satisfying the condition
  
### Sweep line
This category of problem requires sorting a set of operations, then sweeping a line from left to right across them — the most typical examples are computing the union area or union perimeter of rectangles:
- [HDU 1542 Atlantis](http://acm.hdu.edu.cn/showproblem.php?pid=1542) update: range add/subtract, query: take the root node's value directly
- [HDU 1828 Picture](http://acm.hdu.edu.cn/showproblem.php?pid=1828) update: range add/subtract, query: take the root node's value directly


- [(hard)218The Skyline Problem](https://leetcode.cn/problems/the-skyline-problem/)
- [(hard)327Count of Range Sum](https://leetcode.cn/problems/count-of-range-sum/)
- [(hard)699Falling Squares](https://leetcode.cn/problems/falling-squares/)
- [(hard)715Range Module](https://leetcode.cn/problems/range-module/)
- [(medium)729My Calendar I](https://leetcode.cn/problems/my-calendar-i/)
- [(medium)731My Calendar II](https://leetcode.cn/problems/my-calendar-ii/)
- [(hard)732My Calendar III](https://leetcode.cn/problems/my-calendar-iii/)
- [(medium)1109Corporate Flight Bookings](https://leetcode.cn/problems/corporate-flight-bookings/)
- [(hard)1157Online Majority Element In Subarray](https://leetcode.cn/problems/online-majority-element-in-subarray/)
- [(hard)2213Longest Substring of One Repeating Character](https://leetcode.cn/problems/longest-substring-of-one-repeating-character/)

In [ ]:
# 933 Approach 1: Queue
from collections import deque
class RecentCounter:
    def __init__(self):
        self.records = deque()


    def ping(self, t: int) -> int:
        self.records.append(t)
        while self.records[0] < t - 3000:
            self.records.popleft()
        return len(self.records)

In [ ]:
# 307 Approach 1: direct summation, TLE
class NumArray:

    def __init__(self, nums: List[int]):
        self.nums = nums

    def update(self, index: int, val: int) -> None:
        self.nums[index] = val

    def sumRange(self, left: int, right: int) -> int:
        return sum(self.nums[left:right + 1])
    
# Optimized version of Approach 1
class NumArray:

    def __init__(self, nums: List[int]):
        self.nums = nums 
        self.total = sum(self.nums)
        
    def update(self, index: int, val: int) -> None:
        pre_val, self.nums[index] = self.nums[index], val
        self.total = self.total - pre_val + val
        
    def sumRange(self, left: int, right: int) -> int:
        return sum(self.nums[left:right+1]) if right-left < len(self.nums)//2 else self.total - sum(self.nums[:left]) - sum(self.nums[right+1:])
    
# Approach 2: prefix sums, TLE
class NumArray:
    def __init__(self, nums: List[int]):
        self.nums = nums
        self.pre_sum = [0]
        for i in nums:
            self.pre_sum.append(self.pre_sum[-1] + i)

    def update(self, index: int, val: int) -> None:
        diff = val - self.nums[index]
        self.nums[index] = val
        for i in range(index + 1, len(self.nums) + 1):
            self.pre_sum[i] += diff

    def sumRange(self, left: int, right: int) -> int:
        return self.pre_sum[right + 1] - self.pre_sum[left]

# Approach 3: Binary Indexed Tree
'''BIT: Binary Indexed Tree'''
class NumArray:
    
    '''custom lowbit function'''
    def lowbit(self, x: int) -> int:
        return x&(-x)

    def __init__(self, nums: List[int]):
        self.tree = [0] + nums              # the constructed BIT has one more slot than nums; index 0 is unused
        for i in range(1, len(self.tree)):  # building the BIT this way has time complexity O(n)
            j = i + self.lowbit(i)          # the clever trick for building the BIT
            if j < len(self.tree):
                self.tree[j] += self.tree[i]

    def update(self, index: int, val: int) -> None:
        pre_val = self.sumRange(index, index)   # index: position in the original nums
        delta = val - pre_val           # the change in value
        i = index + 1                   # i: the position of this value within the BIT (index+1)
        while i < len(self.tree):
            self.tree[i] += delta
            i += self.lowbit(i)

    '''custom prefix-sum function preSum'''
    def preSum(self, index: int) -> int:    # index: position in the original nums
        i = index + 1                       # i: the position of this value within the BIT (index+1)
        summ = 0
        while i:
            summ += self.tree[i]
            i -= self.lowbit(i)
            # i &= i-1        # equivalent to the line above
        return summ
    
    def sumRange(self, left: int, right: int) -> int:
        return self.preSum(right) - self.preSum(left-1)   # prefix sum at right minus prefix sum at left-1



# Your NumArray object will be instantiated and called as such:
# obj = NumArray(nums)
# obj.update(index,val)
# param_2 = obj.sumRange(left,right)

# Approach 4: Segment tree
class NumArray:

    def __init__(self, nums: List[int]):
        self.len = len(nums)
        self.tree = [None] * 4 * self.len
        self.nums = nums
        self.nid2tid = [None] * self.len
        self.build(0, 0, self.len-1)
        # print(self.tree)
        # print(self.nid2tid)
    
    def build(self, i, l, r):
        if l == r:
            self.tree[i] = self.nums[l]
            self.nid2tid[l] = i
            return 
        mid = (l + r ) // 2
        self.build(2 * i + 1, l, mid)
        self.build(2 * i + 2, mid + 1, r)
        self.tree[i] = self.tree[2*i+1] + self.tree[2*i+2]

    def query(self, tid, left, right, tleft, tright):
        # print("Querying: ", left, right, "for the treeid: ", tid, " within: ", tleft, tright)
        if left == tleft and right == tright:
            return self.tree[tid]
        tmid = (tleft + tright) // 2
        if right <= tmid:
            return self.query(2*tid + 1, left, right, tleft, tmid)
        if left >= tmid + 1:
            return self.query(2*tid + 2, left, right, tmid + 1, tright)
        return self.query(2*tid + 1, left, tmid, tleft, tmid) + self.query(2*tid + 2, tmid + 1, right, tmid + 1, tright)

    def update(self, index: int, val: int) -> None:
        diff = val - self.nums[index]
        nid = self.nid2tid[index]
        while nid >= 0:
            self.tree[nid] += diff
            nid = (nid + 1) // 2 - 1
        # print("Tree after updating: ", index, self.nums[index], val, self.tree)
        self.nums[index] = val
        
    def sumRange(self, left: int, right: int) -> int:
        return self.query(0, left, right, 0, self.len-1)
        


# Your NumArray object will be instantiated and called as such:
# obj = NumArray(nums)
# obj.update(index,val)
# param_2 = obj.sumRange(left,right)

In [ ]:
# 315 Approach 1: Nested loops
class Solution:
    def countSmaller(self, nums: List[int]) -> List[int]:
        res = [] 
        for i, num in enumerate(nums):
            cnt = 0
            for j in range(i+1, len(nums)):
                if nums[j] < num:
                    cnt += 1
            res.append(cnt)
        return res
    
# Approach 2: reverse traversal + binary search + sorted insertion, time 3012ms, space 31.1MB
class Solution:
    def __init__(self):
        self.sorted = []

    def getCurID(self, num):
        l = 0
        r = len(self.sorted) - 1
        if num > self.sorted[-1]:
            return r + 1
        if num <= self.sorted[0]:
            return l
        while l < r:
            mid = (l+r)//2
            if self.sorted[mid] >= num:
                r = mid
            else:
                l = mid + 1
        return l

    def findSmallerNumber(self, num):
        if not self.sorted:
            self.sorted.append(num)
            return 0
        cur_id = self.getCurID(num)
        self.sorted.insert(cur_id, num)
        return cur_id

    def countSmaller(self, nums: List[int]) -> List[int]:
        res = []
        ttl = len(nums)
        for i in range(ttl - 1, -1, -1):
            smaller_num = self.findSmallerNumber(nums[i])
            res.append(smaller_num)
        return res[::-1]
    
# Approach 2: using Python's built-in sorted data structure
from sortedcontainers import SortedList
class Solution:
    def countSmaller(self, nums: List[int]) -> List[int]:

        n = len(nums)
        res = [0] * n
        sl = SortedList()

        for i in range(n-1, -1, -1):        # traverse in reverse
            cnt = sl.bisect_left(nums[i])   # find the count of elements to the right that are smaller than the current value
            res[i] = cnt                    # record into the answer
            sl.add(nums[i])                 # insert the current value into the sorted structure
        
        return res
    
# Approach 4: Binary Indexed Tree, time 1712ms, space 33.4MB
class Solution:
    def __init__(self):
        self.tree = []
        self.n = 0

    def lowbit(self, x):
        return x & (-x)

    def buildTree(self, n):
        self.n = n + 1
        self.tree = [0] * (n + 1)

    def query(self, i):
        pre_sum = 0
        tid = i + 1 # note: need to add 1
        while tid > 0: # note: strictly greater than 0, not greater-or-equal
            pre_sum += self.tree[tid]
            tid -= self.lowbit(tid)
        return pre_sum

    def add(self, i, val):
        tid = i + 1 # note: need to add 1
        while tid < self.n:
            self.tree[tid] += val
            tid += self.lowbit(tid)

    def countSmaller(self, nums: List[int]) -> List[int]:
        res = []
        uniques = sorted(set(nums))
        ranks = {num: rank for rank, num in enumerate(uniques)} # small trick: coordinate compression
        self.buildTree(len(uniques))
        for i in range(len(nums) -1, -1, -1):
            ranki = ranks[nums[i]]
            res.append(self.query(ranki - 1))
            self.add(ranki, 1)
        return res[::-1]
    
# Approach 5: Segment tree, time 3248ms, space 35.4MB
class Solution:
    def __init__(self):
        self.tree = []
        self.r = 0
        self.nid2tid = {}


    def buildTree(self, n):
        self.tree = [0] * 4 * n
        self.r = n - 1
        self.recurBuild(0, 0, n - 1)

    def recurBuild(self, tid, l, r):
        if l == r:
            self.nid2tid[l] = tid
            return
        m = (l + r) // 2
        self.recurBuild(2*tid + 1, l, m)
        self.recurBuild(2*tid + 2, m + 1, r)

    def query(self, r):
        if r < 0:
            return 0
        return self.queryInTree(0, 0, self.r, 0, r)
        
    def queryInTree(self, tid, tleft, tright, qleft, qright):
        if tleft == qleft and tright == qright:
            return self.tree[tid]
        tmid = (tleft + tright) // 2
        if qleft >= tmid + 1:
            return self.queryInTree(2*tid + 2, tmid + 1, tright, qleft, qright)
        if qright <= tmid:
            return self.queryInTree(2*tid+1, tleft, tmid, qleft, qright)
        return self.queryInTree(2*tid+1, tleft, tmid, qleft, tmid) + self.queryInTree(2*tid + 2, tmid + 1, tright, tmid + 1, qright)

    def add(self, i, val):
        tid = self.nid2tid[i]
        while tid >= 0:
            self.tree[tid] += val
            tid = (tid + 1) // 2 - 1

    def countSmaller(self, nums: List[int]) -> List[int]:
        res = []
        uniques = sorted(set(nums))
        ranks = {num: rank for rank, num in enumerate(uniques)}
        self.buildTree(len(uniques))
        for i in range(len(nums) -1, -1, -1):
            ranki = ranks[nums[i]]
            res.append(self.query(ranki - 1))
            self.add(ranki, 1)
        return res[::-1]
    
# Approach 5: official-solution version, time 1896ms, space 32.2MB
''''ST: Segment Tree'''
class ST:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (2*n)     # the number of elements is twice that of the original array
    
    def add(self, i, delta):
        i += self.n                 # convert the original array index to the segment tree index
        while i>0:
            self.tree[i] += delta
            i //= 2

    def rangeSum(self, i, j):       # range sum 【accumulated bottom-up】
        i, j = i+self.n, j+self.n   # convert the original array indices to segment tree indices
        summ = 0
        while i<=j:
            if i&1 == 1:        # right child
                summ += self.tree[i]
                i += 1
            if j&1 == 0:        # left child
                summ += self.tree[j]
                j -= 1
            i //= 2
            j //= 2
        return summ


class Solution:
    def countSmaller(self, nums: List[int]) -> List[int]:

        n = len(nums)
        
        # coordinate compression: convert absolute values to ranks 【rank starts at 0】
        uniques = sorted(set(nums))
        rank_map = {v:i for i,v in enumerate(uniques)}
        
        # build the segment tree
        tree = ST(len(uniques))
        
        # query from right to left
        res = [0] * n
        for i in range(n-1, -1, -1):
            rank = rank_map[nums[i]]    # the rank of the current value
            tree.add(rank, 1)           # single-point update +1
            res[i] = tree.rangeSum(0, rank-1)    # query how many elements come before the current rank

        return res


# Summary

From Gong Shuisanye:

For different problem types, we have different options to choose from (assuming we have an array):

    Array never changes, query range sum: "prefix sum", "Binary Indexed Tree", "segment tree"
    Multiple single-point updates, query range sum: "Binary Indexed Tree", "segment tree"
    Multiple range updates, output the final result: "difference array"
    Multiple range updates, query range sum: "segment tree", "Binary Indexed Tree" (depends on the size of the updated range)
    Multiple range-assignments to a single value, query range sum: "segment tree", "Binary Indexed Tree" (depends on the size of the updated range)

Looking at this, the "segment tree" can solve the most types of problems — so should we just always write a "segment tree" regardless of the situation?

The answer is no — in fact, quite the opposite. We only consider the "segment tree" when we run into the 4th category of problem, where we have no choice but to write one.

Because "segment tree" code is long and has a large constant factor, its real-world performance isn't great. We only consider a "segment tree" when we have no other choice.

To summarize, we should consider things in this order of priority:

- For a simple range-sum query, use "prefix sum"
- For multiple range-assignments to a single value, use "segment tree"
- For other cases, use "Binary Indexed Tree"

Some points about segment trees worth thinking about:
- How to build the tree? Bottom-up? Top-down? What I want to know is the coverage range of each node
- What properties does the merge function need to satisfy?

Using the BIT's prefix sums: can be used to track some statistic of the array up to its current state. The advantage of a BIT: convenient updates and queries.
- When implementing a Binary Indexed Tree, be careful about:
    - Adding one extra element at the front — the BIT's length is one more than the original array's.
    - Array indices actually start from 1. For each queried index i, its position in the BIT array is tid=i+1